RANDOM FOREST REGRESSOR MODEL

Michael Owens 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

In [ ]:
# Bringing in the Data

# read data
df = pd.read_csv('C:/Users/owensmt/OneDrive - Rose-Hulman Institute of Technology/Documents/cssema415/data/data/Concrete/Concrete_train.csv')
print(df.shape)
df_test_set = pd.read_csv('C:/Users/owensmt/OneDrive - Rose-Hulman Institute of Technology/Documents/cssema415/data/data/Concrete/Concrete_test.csv')
df.head(3)

Grid Search to Determine the Number of Estimators and Tree Depth

In [ ]:
#Split data & define features/targets
(df_train,df_test) = train_test_split(df,train_size=0.8,
                                      test_size=0.2,
                                      random_state=0)
X_train = df_train.drop('Concrete compressive strength(MPa, megapascals) ',axis=1)
X_test  = df_test.drop('Concrete compressive strength(MPa, megapascals) ',axis=1)
y_train   = df_train['Concrete compressive strength(MPa, megapascals) ']
y_test    = df_test['Concrete compressive strength(MPa, megapascals) ']

In [ ]:
grid = {'max_depth' : np.arange(1,20,3),'n_estimators':np.arange(1,2500,250)}
rfr = RandomForestRegressor(max_features = 1/3)
rfrCV = GridSearchCV(rfr,param_grid= grid,n_jobs=-1)
rfrCV.fit(X_train,y_train)
print('Random Forest Regressor:')
print('    Optimal Parameters:', rfrCV.best_params_ )
print('    Optimal Valid R2 =', rfrCV.best_score_ )

Implementing a Heap Map to Assist Grid Search

In [ ]:
Scores_mean = rfrCV.cv_results_['mean_test_score']
Scores_mean = Scores_mean.reshape(len(grid['max_depth']),len(grid['n_estimators']))

In [ ]:
fig = px.imshow(Scores_mean,labels=dict(x='Tree Depth',y='Number of Estimators'),y=grid['max_depth'],x=grid['n_estimators'],aspect='auto')
fig.show()

Refined Grid Search

In [ ]:
grid = {'max_depth' : np.arange(17,24,1),'n_estimators':np.arange(485,515,1)}
rfr = RandomForestRegressor(max_features = 1/3)
rfrCV = GridSearchCV(rfr,param_grid= grid, oob_score=True,n_jobs=-1)
rfrCV.fit(X_train,y_train)
print('Random Forest Regressor:')
print('    Optimal Parameters:', rfrCV.best_params_ )
print('    Optimal Valid R2 =', rfrCV.best_score_ )

Scores_mean = rfrCV.cv_results_['mean_test_score']
Scores_mean = Scores_mean.reshape(len(grid['max_depth']),len(grid['n_estimators']))

fig = px.imshow(Scores_mean,labels=dict(x='Tree Depth',y='Number of Estimators'),y=grid['max_depth'],x=grid['n_estimators'],aspect='auto')
fig.show()

Evaluating the Model

In [ ]:
results = pd.DataFrame()
results['trees'] = grid['n_estimators']
results['train R2'] = rfrCV.cv_results_['mean_train_score']
results['valid R2']  = rfrCV.cv_results_['mean_test_score']
results[['train R2','valid R2']].plot.line()

In [ ]:
# test R2
print(f"test R2 {rfrCV.score(X_test,y_test):.3f}")

MSE